# torch-aac Demo

**GPU-accelerated, differentiable AAC-LC encoder in PyTorch.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VerdaAI/torch-aac/blob/main/examples/demo.ipynb)
[![GitHub](https://img.shields.io/github/stars/VerdaAI/torch-aac?style=social)](https://github.com/VerdaAI/torch-aac)

This notebook demonstrates:
1. **Encoding** — PCM audio → AAC → decode and compare
2. **Quality** — SNR benchmarks across signal types
3. **Differentiable mode** — gradients flow through AAC for training
4. **Speed** — realtime factor measurement

In [ ]:
# Install (takes ~30s on Colab)
!pip install -q torch-aac
!apt-get -qq install ffmpeg > /dev/null 2>&1

import torch
import numpy as np
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

import torch_aac
print(f'torch-aac: {torch_aac.__version__}')

## 1. Basic Encoding

Encode a sine wave to AAC and decode it back.

In [ ]:
import subprocess, tempfile
from pathlib import Path
from IPython.display import Audio, display

sr = 48000
duration = 3.0
t = np.arange(int(sr * duration), dtype=np.float32) / sr

# Generate a chord: A4 + E5 + A5
pcm = (0.3 * np.sin(2*np.pi*440*t) +
       0.2 * np.sin(2*np.pi*659*t) +
       0.15 * np.sin(2*np.pi*880*t)).astype(np.float32)

# Encode to AAC
device = 'cuda' if torch.cuda.is_available() else 'cpu'
aac_bytes = torch_aac.encode(pcm, sample_rate=sr, bitrate=128000, device=device)
print(f'Input:   {len(pcm)} samples ({duration:.1f}s)')
print(f'Encoded: {len(aac_bytes)} bytes ({len(aac_bytes)*8/duration/1000:.0f} kbps)')
print(f'Device:  {device}')

# Decode with FFmpeg
with tempfile.NamedTemporaryFile(suffix='.aac', delete=False) as f:
    f.write(aac_bytes); p = f.name
r = subprocess.run(
    ['ffmpeg', '-y', '-v', 'error', '-i', p,
     '-f', 'f32le', '-ar', str(sr), '-ac', '1', 'pipe:1'],
    capture_output=True)
Path(p).unlink()
decoded = np.frombuffer(r.stdout, dtype=np.float32)

# Compare
n = min(len(pcm), len(decoded))
ratio = np.abs(decoded).max() / np.abs(pcm).max()
print(f'\nAmplitude ratio: {ratio:.4f} (1.0 = perfect)')

print('\nOriginal:')
display(Audio(pcm, rate=sr))
print('Decoded from AAC:')
display(Audio(decoded, rate=sr))

## 2. Quality Benchmarks

Measure SNR across different signal types at 128 kbps.

In [ ]:
def measure_snr(orig, dec, skip=1024):
    n = min(len(orig), len(dec))
    o = orig[skip:n].astype(np.float64)
    d = dec[skip:n].astype(np.float64)
    orms = np.sqrt(np.mean(o**2))
    drms = np.sqrt(np.mean(d**2))
    if drms < 1e-10 or orms < 1e-10:
        return -99.0
    dn = d * orms / drms
    noise = np.sqrt(np.mean((dn - o)**2))
    return 20 * np.log10(orms / (noise + 1e-20))

def encode_decode(pcm, sr=48000, bitrate=128000):
    aac = torch_aac.encode(pcm, sample_rate=sr, bitrate=bitrate, device=device)
    with tempfile.NamedTemporaryFile(suffix='.aac', delete=False) as f:
        f.write(aac); p = f.name
    r = subprocess.run(
        ['ffmpeg', '-y', '-v', 'error', '-i', p,
         '-f', 'f32le', '-ar', str(sr), '-ac', '1', 'pipe:1'],
        capture_output=True)
    Path(p).unlink()
    return np.frombuffer(r.stdout, dtype=np.float32), len(aac)

# Test signals
rng = np.random.RandomState(42)
signals = {
    'Sine 1kHz':  (0.5 * np.sin(2*np.pi*1000*t)).astype(np.float32),
    'Chord':      pcm,  # reuse from above
    'White noise': (0.3 * rng.randn(len(t))).astype(np.float32),
    'Sweep':      (0.5 * np.sin(2*np.pi*100 * np.exp(np.log(10000/100)*t/duration) * t / (np.log(10000/100)/duration))).astype(np.float32),
}

print(f'{"Signal":15} {"SNR (dB)":>10} {"Size":>8} {"Ratio":>8}')
print('-' * 45)
for name, sig in signals.items():
    dec, sz = encode_decode(sig)
    snr = measure_snr(sig, dec)
    r = np.abs(dec).max() / np.abs(sig).max()
    print(f'{name:15} {snr:9.1f}  {sz:7d}  {r:7.4f}')

## 3. Differentiable Mode

Gradients flow through the AAC simulation — you can train models to be codec-robust.

In [ ]:
import torch.nn.functional as F

# Create differentiable codec
codec = torch_aac.DifferentiableAAC(
    sample_rate=48000, bitrate=128000,
    quant_mode='ste',  # straight-through estimator
    device=device
)

# Optimize an input to survive AAC compression
target = torch.sin(2*torch.pi*440*torch.arange(4096, dtype=torch.float32, device=device)/48000) * 0.5
x = torch.randn(4096, device=device, requires_grad=True)
optimizer = torch.optim.Adam([x], lr=0.01)

losses = []
for step in range(100):
    optimizer.zero_grad()
    decoded = codec(x)
    loss = F.mse_loss(decoded, target)
    loss.backward()
    optimizer.step()
    if step % 10 == 0:
        losses.append(loss.item())
        print(f'Step {step:3d}: loss = {loss.item():.6f}')

print(f'\nLoss reduced {losses[0]/losses[-1]:.1f}x — gradients work!')

In [ ]:
# Rate-distortion training: also penalize hard-to-compress outputs
decoded, rate_loss = codec(x, return_rate_loss=True)
total_loss = F.mse_loss(decoded, target) + 0.1 * rate_loss
total_loss.backward()
print(f'Rate loss: {rate_loss.item():.4f} (1.0 = budget fully used)')
print(f'Gradient norm: {x.grad.norm().item():.4f}')

## 4. Speed Benchmark

In [ ]:
import time

durations = [1, 5, 10]
print(f'Device: {device}')
print(f'{"Duration":>10} {"Time":>10} {"Realtime":>10}')
print('-' * 35)

for dur in durations:
    audio = (0.5 * np.sin(2*np.pi*440*np.arange(dur*sr)/sr)).astype(np.float32)
    # Warmup
    for _ in range(3):
        torch_aac.encode(audio, sample_rate=sr, bitrate=128000, device=device)
    if device == 'cuda':
        torch.cuda.synchronize()
    # Time
    times = []
    for _ in range(5):
        if device == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        torch_aac.encode(audio, sample_rate=sr, bitrate=128000, device=device)
        if device == 'cuda': torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    best = min(times)
    print(f'{dur:9d}s {best*1000:9.1f}ms {dur/best:9.1f}x')

## What's Next?

- **Try your own audio**: Upload a WAV file and encode it
- **Train a model**: Use `DifferentiableAAC` in your training loop
- **Batch encoding**: `torch_aac.encode_batch([audio1, audio2, ...])` for parallel processing

Full docs: [github.com/VerdaAI/torch-aac](https://github.com/VerdaAI/torch-aac)